In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wasserstein_distance, entropy, chisquare
from itertools import combinations


# 1. CSV-Dateien einlesen (Pfad zu realen und synthetischen Daten anpassen)
real_path = "../../data/data_for_CTGAN/CTGAN_basedata.csv"
syn_path = "../../data/synthetic_result_data/536_synthetic_train_data.csv"
df_real = pd.read_csv(real_path)
df_syn = pd.read_csv(syn_path)


In [2]:
def compute_subscores(original: pd.DataFrame, synthetic: pd.DataFrame, alpha_kl=1.0, mmd_gamma=1.0, weights=None):
    """
    Compute Statistical Similarity Score (SSS) subscores for each feature and overall score.
    Returns a dict with per-feature subscores and the overall SSS.
    """
    def compute_mmd(x, y, gamma):
        """Compute unbiased MMD with RBF kernel, returning a scalar."""
        X = x.values.flatten()
        Y = y.values.flatten()
        Kxx = np.exp(-gamma * (X[:, None] - X[None, :])**2)
        Kyy = np.exp(-gamma * (Y[:, None] - Y[None, :])**2)
        Kxy = np.exp(-gamma * (X[:, None] - Y[None, :])**2)
        n, m = len(X), len(Y)
        mmd2 = (
            (Kxx.sum() - np.trace(Kxx)) / (n * (n - 1))
            + (Kyy.sum() - np.trace(Kyy)) / (m * (m - 1))
            - 2 * Kxy.sum() / (n * m)
        )
        return np.sqrt(max(mmd2, 0))

    # Sprechende Metriknamen
    metric_names = ['mean','median','variance','ks','chi2','wasserstein','mmd','kl','coverage']

    if weights is None:
        # Default: gleichmäßige Verteilung
        weights = {name: 1/len(metric_names) for name in metric_names}

    results = {}
    feature_scores = []

    for col in original.columns:
        o = original[col]
        s = synthetic[col]
        subscores = {}

        if pd.api.types.is_numeric_dtype(o):
            R = o.max() - o.min()
            delta_mean = abs(o.mean() - s.mean())
            subscores['mean'] = max(0, 1 - delta_mean / R) if R > 0 else 1.0

            delta_median = abs(o.median() - s.median())
            subscores['median'] = max(0, 1 - delta_median / R) if R > 0 else 1.0

            V = max(o.var(), s.var())
            delta_var = abs(o.var() - s.var())
            subscores['variance'] = max(0, 1 - delta_var / V) if V > 0 else 1.0

            subscores['ks'] = ks_2samp(o, s).pvalue

            w_dist = wasserstein_distance(o, s)
            subscores['wasserstein'] = 1 / (1 + w_dist)

            mmd_val = compute_mmd(o, s, mmd_gamma)
            subscores['mmd'] = 1 / (1 + mmd_val)

            bins = min(50, max(o.nunique(), s.nunique()))
            p_hist, _ = np.histogram(o, bins=bins, density=True)
            q_hist, _ = np.histogram(s, bins=bins, density=True)
            p_hist += 1e-8
            q_hist += 1e-8
            kl_div = entropy(p_hist, q_hist)
            subscores['kl'] = np.exp(-alpha_kl * kl_div)

            cov = len(np.intersect1d(o.unique(), s.unique())) / o.nunique() if o.nunique() > 0 else 1.0
            subscores['coverage'] = cov

        else:
            real_counts = o.value_counts().sort_index()
            syn_counts = s.value_counts().reindex(real_counts.index, fill_value=0)
            stat, p_value = chisquare(f_obs=syn_counts, f_exp=real_counts)
            subscores['chi2'] = p_value
            cov = len(set(o.unique()) & set(s.unique())) / len(o.unique()) if o.nunique() > 0 else 1.0
            subscores['coverage'] = cov

        # Nur vorhandene Metriken werten
        available_metrics = [k for k in subscores if k in weights]
        total_weight = sum(weights[k] for k in available_metrics)

        score = sum(weights[k] * subscores[k] for k in available_metrics) / total_weight if total_weight > 0 else 0
        results[col] = {'subscores': subscores, 'feature_score': score}
        feature_scores.append(score)

    overall_sss = float(np.mean(feature_scores))
    return {'per_feature': results, 'SSS': overall_sss}


In [ ]:
custom_weights = {
    'mean': 0.05,
    'median': 0.05,
    'variance': 0.05,
    'ks': 0.1,
    'chi2': 0.1,
    'wasserstein': 0.2,
    'mmd': 0.2,
    'kl': 0.2,
    'coverage': 0.05
}


result = compute_subscores(df_real, df_syn, weights=custom_weights)
print("Statistical Similarity Score (SSS):", result['SSS'])


for feature, data in result['per_feature'].items():
    print(f"\nFeature: {feature}")
    print(" Subscores:", data['subscores'])
    print(" Feature score:", data['feature_score'])



Statistical Similarity Score (SSS): 0.781359304632074

Feature: geschlecht
 Subscores: {'chi2': np.float64(6.612374506157755e-29), 'coverage': 1.0}
 Feature score: 0.3333333333333333

Feature: alter
 Subscores: {'chi2': np.float64(6.59652050293519e-14), 'coverage': 0.9333333333333333}
 Feature score: 0.3111111111111551

Feature: bundesland
 Subscores: {'chi2': np.float64(4.792397314701809e-05), 'coverage': 0.46153846153846156}
 Feature score: 0.15387810316158518

Feature: beruf
 Subscores: {'chi2': np.float64(1.3227155700653558e-38), 'coverage': 0.8888888888888888}
 Feature score: 0.2962962962962963

Feature: konsumhaeufigkeit
 Subscores: {'chi2': np.float64(0.001333341076843179), 'coverage': 1.0}
 Feature score: 0.3342222273845621

Feature: situation_freunde_familie
 Subscores: {'mean': np.float64(0.9365671641791047), 'median': np.float64(0.5), 'variance': np.float64(1.0), 'ks': np.float64(0.0003530494711383479), 'wasserstein': np.float64(0.8874172185430463), 'mmd': np.float64(0.87867